# Results Visualization

Comprehensive visualizations for model performance analysis:
- Performance comparison charts
- Feature importance plots
- Error analysis
- Learning curves
- Prediction distribution analysis

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os, json
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Libraries loaded.')
from sklearn.learning_curve import learning_curve

In [ ]:
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'Dataset: {df.shape[0]} rows, Target positive rate: {df["target"].mean()*100:.1f}%')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower()
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)
print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty: {(df["cleaned_message"]!="").sum()}')

In [ ]:
df['response_time_minutes'] = ((df['issue_responded']-df['issue_reported_at']).dt.total_seconds()/60).clip(lower=0).fillna(0)
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)
for col,src in [('channel_encoded','channel_name'),('category_encoded','category'),
                ('subcategory_encoded','sub-category'),('shift_encoded','agent_shift')]:
    le=LabelEncoder(); df[col]=le.fit_transform(df[src].fillna('Unknown'))
tenure_map={'On Job Training':0,'0-30':1,'31-60':2,'61-90':3,'>90':4}
df['tenure_encoded']=df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)
df['has_message']=(df['cleaned_message']!='').astype(int)
df['cleaned_word_count']=df['cleaned_message'].apply(lambda x:len(x.split()) if x else 0)
structured_features=['response_time_minutes','issue_hour','issue_day_of_week','channel_encoded',
    'category_encoded','subcategory_encoded','shift_encoded','tenure_encoded',
    'message_length','word_count','has_message','cleaned_word_count']
print(f'Structured features: {len(structured_features)}')

In [ ]:
X_structured = df[structured_features].fillna(0)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_structured, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_text_train = tfidf.fit_transform(df.loc[X_train.index,'cleaned_message'])
X_text_test = tfidf.transform(df.loc[X_test.index,'cleaned_message'])
X_combined_train = hstack([X_text_train, csr_matrix(X_train.values)])
X_combined_test = hstack([X_text_test, csr_matrix(X_test.values)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg_count=(y_train==0).sum(); pos_count=(y_train==1).sum(); scale_weight=neg_count/pos_count
print(f'Train:{X_train.shape[0]}, Test:{X_test.shape[0]}, TF-IDF:{X_text_train.shape[1]}, Combined:{X_combined_train.shape[1]}')

## Section 2: Train Models for Visualization

In [ ]:
# Train models
lr=LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced');lr.fit(X_combined_train,y_train)
rf=RandomForestClassifier(n_estimators=200,max_depth=20,random_state=42,class_weight='balanced',n_jobs=-1);rf.fit(X_text_train,y_train)
xgb=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False);xgb.fit(X_train,y_train)
gb=GradientBoostingClassifier(n_estimators=200,max_depth=5,random_state=42);gb.fit(X_train,y_train)
svm=CalibratedClassifierCV(LinearSVC(max_iter=2000,random_state=42,class_weight='balanced'),cv=3);svm.fit(X_text_train,y_train)
print('All models trained.')

## Section 3: Performance Comparison Heatmap

In [ ]:
# Metrics heatmap
models_eval={'LR':(lr.predict(X_combined_test),lr.predict_proba(X_combined_test)[:,1]),
    'RF':(rf.predict(X_text_test),rf.predict_proba(X_text_test)[:,1]),
    'XGB':(xgb.predict(X_test),xgb.predict_proba(X_test)[:,1]),
    'GB':(gb.predict(X_test),gb.predict_proba(X_test)[:,1]),
    'SVM':(svm.predict(X_text_test),svm.predict_proba(X_text_test)[:,1])}

metrics_matrix=[]
for name,(pred,prob) in models_eval.items():
    metrics_matrix.append([accuracy_score(y_test,pred),precision_score(y_test,pred),
        recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,prob)])
metrics_df=pd.DataFrame(metrics_matrix,index=models_eval.keys(),columns=['Accuracy','Precision','Recall','F1','AUC-ROC'])

fig,ax=plt.subplots(figsize=(10,6))
sns.heatmap(metrics_df,annot=True,fmt='.3f',cmap='YlOrRd',ax=ax,vmin=0.5,vmax=1.0,linewidths=0.5)
ax.set_title('Model Performance Heatmap',fontsize=14)
plt.tight_layout();plt.savefig('../models/performance_heatmap.png',dpi=150,bbox_inches='tight');plt.show()

## Section 4: Feature Importance

In [ ]:
# Feature importance comparison
fig,axes=plt.subplots(1,2,figsize=(14,6))

# XGBoost feature importance
xgb_imp=pd.DataFrame({'feature':structured_features,'importance':xgb.feature_importances_}).sort_values('importance',ascending=True)
axes[0].barh(xgb_imp['feature'],xgb_imp['importance'],color='steelblue')
axes[0].set_title('XGBoost Feature Importance')

# Gradient Boosting feature importance
gb_imp=pd.DataFrame({'feature':structured_features,'importance':gb.feature_importances_}).sort_values('importance',ascending=True)
axes[1].barh(gb_imp['feature'],gb_imp['importance'],color='coral')
axes[1].set_title('Gradient Boosting Feature Importance')

plt.tight_layout();plt.savefig('../models/feature_importance.png',dpi=150,bbox_inches='tight');plt.show()

## Section 5: Learning Curves

In [ ]:
# Learning curves
fig,axes=plt.subplots(1,3,figsize=(16,5))
lc_models=[('XGBoost',XGBClassifier(n_estimators=100,max_depth=6,learning_rate=0.1,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False),X_train),
    ('Gradient Boosting',GradientBoostingClassifier(n_estimators=100,max_depth=5,random_state=42),X_train),
    ('Logistic Regression',LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced'),X_combined_train)]

for i,(name,model,X_lc) in enumerate(lc_models):
    train_sizes,train_scores,val_scores=learning_curve(model,X_lc,y_train,cv=5,scoring='f1',
        train_sizes=np.linspace(0.1,1.0,10),n_jobs=-1)
    axes[i].plot(train_sizes,train_scores.mean(axis=1),'o-',label='Train',color='steelblue')
    axes[i].fill_between(train_sizes,train_scores.mean(axis=1)-train_scores.std(axis=1),
        train_scores.mean(axis=1)+train_scores.std(axis=1),alpha=0.1,color='steelblue')
    axes[i].plot(train_sizes,val_scores.mean(axis=1),'o-',label='Validation',color='coral')
    axes[i].fill_between(train_sizes,val_scores.mean(axis=1)-val_scores.std(axis=1),
        val_scores.mean(axis=1)+val_scores.std(axis=1),alpha=0.1,color='coral')
    axes[i].set_title(f'{name}');axes[i].set_xlabel('Training Size');axes[i].set_ylabel('F1 Score')
    axes[i].legend();axes[i].grid(alpha=0.3)
plt.suptitle('Learning Curves',fontsize=14);plt.tight_layout()
plt.savefig('../models/learning_curves.png',dpi=150,bbox_inches='tight');plt.show()

## Section 6: Error Analysis

In [ ]:
# Error analysis - where does the best model fail?
y_pred_best=xgb.predict(X_test)
errors=X_test[y_pred_best!=y_test].copy()
correct=X_test[y_pred_best==y_test].copy()
errors['actual']=y_test[y_pred_best!=y_test]
correct['actual']=y_test[y_pred_best==y_test]

fig,axes=plt.subplots(2,2,figsize=(14,10))
# Error by response time
axes[0,0].hist(errors['response_time_minutes'].clip(upper=500),bins=30,alpha=0.7,label='Misclassified',color='coral')
axes[0,0].hist(correct['response_time_minutes'].clip(upper=500),bins=30,alpha=0.5,label='Correct',color='steelblue')
axes[0,0].set_title('Response Time: Errors vs Correct');axes[0,0].legend()

# Error by category
err_cats=errors['category_encoded'].value_counts().head(8)
axes[0,1].bar(err_cats.index.astype(str),err_cats.values,color='coral')
axes[0,1].set_title('Misclassification by Category');axes[0,1].set_xlabel('Category ID')

# Prediction confidence for errors
y_prob_best=xgb.predict_proba(X_test)[:,1]
err_probs=y_prob_best[y_pred_best!=y_test]
axes[1,0].hist(err_probs,bins=20,color='coral',edgecolor='white')
axes[1,0].set_title('Prediction Confidence (Misclassified)');axes[1,0].set_xlabel('P(Positive)')

# Correct vs incorrect confidence
axes[1,1].hist(y_prob_best[y_pred_best==y_test],bins=20,alpha=0.6,label='Correct',color='steelblue')
axes[1,1].hist(err_probs,bins=20,alpha=0.6,label='Error',color='coral')
axes[1,1].set_title('Confidence Distribution');axes[1,1].legend()

plt.tight_layout();plt.savefig('../models/error_analysis.png',dpi=150,bbox_inches='tight');plt.show()
print(f'Total errors: {len(errors)}/{len(y_test)} ({len(errors)/len(y_test)*100:.1f}%)')

## Section 7: Prediction Distribution

In [ ]:
# Prediction probability distributions
fig,axes=plt.subplots(1,2,figsize=(14,5))
for name,(_,prob) in models_eval.items():
    axes[0].hist(prob,bins=30,alpha=0.5,label=name)
axes[0].set_title('Prediction Probability Distribution (All Models)');axes[0].set_xlabel('P(Positive)');axes[0].legend(fontsize=8)

# Calibration-like plot
from sklearn.calibration import calibration_curve
for name,(_,prob) in models_eval.items():
    frac_pos,mean_pred=calibration_curve(y_test,prob,n_bins=10)
    axes[1].plot(mean_pred,frac_pos,'o-',label=name)
axes[1].plot([0,1],[0,1],'k--');axes[1].set_title('Calibration Plot');axes[1].set_xlabel('Mean Predicted');axes[1].set_ylabel('Fraction Positive');axes[1].legend(fontsize=8);axes[1].grid(alpha=0.3)
plt.tight_layout();plt.savefig('../models/prediction_distribution.png',dpi=150,bbox_inches='tight');plt.show()
print('All visualizations saved to ../models/')